# European option (Python bindings)

This notebook tracks the first vertical slice of `sjd` exposed to Python:

- `datetime.date` and lists on the Python side (QuantLib types stay in C++)
- `ForwardCurve`, `DiscountCurve`, `VolatilityCurve`
- `RatesEnvironment`
- `EuropeanOption.get_option_value` / `get_value`

Build the module first (copies `_core.pyd` into `python/sjd`):

`cmake --build --preset debug --target sjd_python`

Install the package once with `python -m pip install -e python`.

Shut down the notebook kernel before rebuilding so Windows can overwrite `_core.pyd`.

In [1]:
import math
from datetime import date

import sjd
from sjd import examples

print(sjd.__file__)
print([name for name in dir(sjd) if not name.startswith("_")])


E:\Code\sjd\python\sjd\__init__.py
['AsianForward', 'AsianOption', 'Black76Call', 'Black76Put', 'BusinessDayConvention', 'BuySell', 'Calendar', 'Contract', 'CurveInterpolator', 'DailyFixingCurve', 'DiscountCurve', 'EuropeanForward', 'EuropeanOption', 'ForwardAveraging', 'ForwardCurve', 'ForwardRoll', 'Margin', 'Portfolio', 'PortfolioB2CEM', 'PortfolioEvolver', 'PutCall', 'RatesEnvironment', 'RegCapCreditParameters', 'RegCapUnderlyingFactors', 'RegulatoryCapital', 'SimpleMargin', 'Simulation', 'SingleFactorEconomicScenarioGenerator', 'SingleInstrumentContract', 'SurfaceInterpolator', 'TimeUnit', 'ValidationIssue', 'ValidationReport', 'ValidationSeverity', 'VolatilityCurve', 'VolatilityCurveCMA', 'VolatilityRateSource', 'VolatilitySurfaceDelta', 'VolatilitySurfaceMoneynessCMA', 'examples', 'reg_cap_factors', 'simulate']


## Market data

Flat NACC discount 6.5%, IR differential 4.5%, spot 101.23, vol 21.05%.  
Year fractions are Actual/365, matching the C++ `VolatilityCurve` binding.

In [2]:
anchor = date(2013, 1, 1)
obs = [date(2013, 1, 1), date(2015, 1, 1), date(2015, 1, 5), date(2023, 1, 1)]

def yf(d):
    return examples.year_fraction(d, anchor)

spot, nacc, ir_diff, vol = 101.23, 0.065, 0.045, 0.2105
forwards = [spot * math.exp(ir_diff * yf(d)) for d in obs]
dfs = [math.exp(-nacc * yf(d)) for d in obs]
vols = [vol] * len(obs)

frs = sjd.ForwardCurve(anchor, obs, forwards, allow_extrapolation=True)
drs = sjd.DiscountCurve(anchor, obs, dfs, allow_extrapolation=True)
vrs = sjd.VolatilityCurve(anchor, obs, vols, allow_extrapolation=True)

print("curves ok:", frs.is_ok(), drs.is_ok(), vrs.is_ok())
print("F", frs.get_forward(date(2015, 1, 3)))
print("df", drs.get_discount_factor(date(2015, 1, 3)))
print("vol", vrs.get_volatility(date(2015, 1, 1)))


curves ok: True True True
F 110.79058096436813
df 0.8777827959653548
vol 0.2105


## Rates environment and option

`get_option_value` is the Black premium times volume.  
`get_value` also subtracts the PV of the 3.4 premium (same convention as the C++ contract).

In [3]:
re = sjd.RatesEnvironment(anchor)
re.set_frs(frs)
re.set_drs(drs)
re.set_vrs(vrs)
print("environment:", re.is_ok(), re.anchor_date())

opt = sjd.EuropeanOption(
    maturity=date(2015, 1, 1),
    settlement=date(2015, 1, 3),
    strike=100.0,
    type=sjd.PutCall.PUT,
    volume=10000.0,
    premium_payment_date=date(2015, 1, 3),
    premium_amount=3.4,
    buy_sell=sjd.BuySell.BUY,
)
opt.validate(re)
print("option ok:", opt.is_ok(), opt.error_messages())
print("option_value", opt.get_option_value(re))
print("value", opt.get_value(re))

environment: True 2013-01-01
option ok: True 
option_value 68476.02928009552
value 38631.41421727346
